# Interactive 3D spline map for the digits dataset

This view keeps the schematic spline network on the **XY plane**. The first local residual PCA coordinate places observations laterally around their assigned spline; the second local residual PCA coordinate becomes **Z**. Rotate and zoom the Plotly figure to inspect the high-dimensional residual structure along the learned routes.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from sklearn.datasets import load_digits

working_dir = Path.cwd().resolve()
notebooks_dir = working_dir / 'notebooks' if (working_dir / 'notebooks' / '__init__.py').exists() else working_dir
project_root = notebooks_dir.parent
if not (project_root / 'topological_graph_embedding').exists():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(notebooks_dir))

from topological_graph_embedding import SplineGraphEmbedding
from topological_graph_embedding.visualization.interactive import plot_spline_3d

## Fit the spline graph

Digits are used only as observations and labels; the graph is fitted without using the digit targets.

In [ ]:
digits = load_digits()
X = np.asarray(digits.data, dtype=float)
y = np.asarray(digits.target)

model = SplineGraphEmbedding(
    n_centroids=36,
    persistence_threshold=None,
    spline_smoothing=0.02,
    max_cycles=4,
    persistence_max_points=60,
    random_state=0,
)
result = model.fit_transform(X)

print({
    'observations': len(X),
    'features': X.shape[1],
    'cycles': model.realized_cycle_count_,
    'junctions': len(model.junctions_),
    'splines': len(model.routes_),
    'median residual': float(np.median(result.residual_norm)),
})

## Rotate the 3D metro map

Point color is the digit target. Hover a point for its spline assignment, longitudinal coordinate, residual norm, and 3D map coordinates. The spline lines are drawn at `z=0`.

In [ ]:
figure = plot_spline_3d(
    model,
    result,
    labels=y,
    title='Digits: spline network with local residual PCA plane',
    z_scale=1.0,
    point_size=3.5,
)
figure.show()